# 01 — EDA

Display only. All logic lives in `src/s6e7/` (CLAUDE.md rule 5).
Every plot below is implemented by hand in `src/s6e7/plots.py`; cells raise
`NotImplementedError` until the corresponding function is written.

Build order is `PLOTS_SPEC.md`: `target_overview` and `missingness` first.

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from s6e7 import eda, io, plots

plt.rcParams["figure.dpi"] = 120
pl.Config.set_tbl_rows(30)

polars.config.Config

In [3]:
train = io.load_train()
test = io.load_test()

train.shape, test.shape

((690088, 15), (295753, 14))

## Step 1 — column overview

**Decision:** which columns are numeric vs categorical, which need imputation, and
which are effectively constant.

`n_unique` here excludes nulls. Polars' own `n_unique` counts null as a distinct value,
which reports a 3-level categorical with missing data as having 4 levels.

In [4]:
eda.overview(train)

column,dtype,nulls,null_pct,n_unique
str,str,i64,f64,i64
"""id""","""UInt32""",0,0.0,690088
"""health_condition""","""String""",0,0.0,3
"""sleep_duration""","""Float32""",75999,11.01,701
"""heart_rate""","""Float32""",7833,1.14,537
"""bmi""","""Float32""",13898,2.01,1596
"""calorie_expenditure""","""Float32""",52853,7.66,2101
"""step_count""","""Float32""",13916,2.02,12807
"""exercise_duration""","""Float32""",6901,1.0,856
"""water_intake""","""Float32""",43477,6.3,400


In [5]:
eda.overview(test)

column,dtype,nulls,null_pct,n_unique
str,str,i64,f64,i64
"""id""","""UInt32""",0,0.0,295753
"""sleep_duration""","""Float32""",32571,11.01,692
"""heart_rate""","""Float32""",3357,1.14,526
"""bmi""","""Float32""",5956,2.01,1548
"""calorie_expenditure""","""Float32""",22652,7.66,2068
"""step_count""","""Float32""",5964,2.02,12196
"""exercise_duration""","""Float32""",2958,1.0,817
"""water_intake""","""Float32""",18633,6.3,392
"""diet_type""","""String""",2958,1.0,3


## Step 2 — category levels

**Decision:** encoding strategy, and whether naive encoders are safe.

A level appearing only in **test** has no encoding learned for it and breaks a fitted
encoder at predict time. A level only in **train** is dead weight.

Levels print alphabetically, which is rarely the meaningful order — `high, low, medium`
sorts nothing like `low < medium < high`. Reading which of these are genuinely *ordinal*
is the judgement this step exists to support.

In [7]:
eda.category_levels(train, io.CATEGORICAL_COLS, test)

column,n_levels,levels,test_only,train_only
str,i64,str,str,str
"""diet_type""",3,"""balanced, non-veg, veg""","""""",""""""
"""stress_level""",3,"""high, low, medium""","""""",""""""
"""sleep_quality""",3,"""average, good, poor""","""""",""""""
"""physical_activity_level""",3,"""active, moderate, sedentary""","""""",""""""
"""smoking_alcohol""",3,"""no, occasional, yes""","""""",""""""
"""gender""",3,"""female, male, other""","""""",""""""


## Step 3 — class rate per level

**Decision:** is the target *monotone* in the declared level ordering?

That is the empirical claim an ordinal integer encoding makes, and the precondition for
a monotone constraint later. Read each column's block top to bottom: if a class rate
moves consistently in one direction, the ordering is real. If it zigzags, a threshold
split cannot isolate the middle level, and one-hot — or native categorical handling —
wins.

Orderings come from `io.ORDINAL_LEVELS`, a **modelling judgement, not a fact**. Edit it
there if you disagree.

Nulls appear as their own level, so this doubles as the missingness test for the
categoricals. Compare every row against the global rates: `at-risk` 0.859,
`unhealthy` 0.084, `fit` 0.058.

In [10]:
eda.level_target_rates(train, io.CATEGORICAL_COLS, io.TARGET, io.ORDINAL_LEVELS)

column,level,rows,share_pct,p_at-risk,p_fit,p_unhealthy
str,str,i64,f64,f64,f64,f64
"""diet_type""","""balanced""",226888,32.88,0.8514,0.0612,0.0874
"""diet_type""","""non-veg""",224867,32.59,0.8678,0.0516,0.0806
"""diet_type""","""veg""",231432,33.54,0.8567,0.0604,0.083
"""diet_type""","""<null>""",6901,1.0,0.8676,0.0509,0.0816
"""stress_level""","""low""",167708,24.3,0.7967,0.2006,0.0028
"""stress_level""","""medium""",261819,37.94,0.9939,0.003,0.0031
"""stress_level""","""high""",177750,25.76,0.7178,0.0035,0.2787
"""stress_level""","""<null>""",82811,12.0,0.8589,0.0575,0.0836
"""sleep_quality""","""poor""",212166,30.74,0.8296,0.0346,0.1358


## Step 4 — numeric summary

**Decision:** which features need transformation, and whether a bound is a real limit or
a clip.

`grid` is the smallest gap between adjacent distinct values. Synthetic data is almost
always quantised; a coarse grid means the column is effectively discrete.

`pct_at_min` / `pct_at_max` expose clipping. A genuine distribution *tapers* at its
extremes; a clipped one *piles up* there. A pile at exactly 0 raises the separate
question of whether 0 means "none" or "not recorded".

In [ ]:
eda.numeric_summary(train, io.NUMERIC_COLS)

## Step 5 — is missingness a feature?

**Decision:** add missing-indicator columns, or don't bother.

If the class rate differs between missing and present, missingness carries information
and deserves an indicator. If the rates match, the nulls were injected at random and an
indicator is dead weight — and clever imputation buys nothing either.

Sorted by `abs_diff`, largest first. The `<null>` rows in step 3 already hint at the
answer for the categoricals.

In [ ]:
eda.missing_vs_target(train, io.FEATURE_COLS, io.TARGET).sort("abs_diff", descending=True).head(12)

In [ ]:
eda.missing_cooccurrence(train, io.FEATURE_COLS).head(10)

## Step 6 — which numeric features carry signal

**Decision:** which features to plot, and what to expect from a baseline.

`spread_sd` is the gap between the largest and smallest class mean, measured in the
column's own standard deviations — a crude effect size. Near zero means the three
classes sit on top of each other and a density plot will show three overlapping curves.

This is the table that decides which of the seven numerics are worth a panel. Build the
density plots **only** for the ones that score here.

Its blind spot: it only compares *means*. Two classes with equal means and different
variances score zero and are still separable — exactly the case where a plot beats a
table.

In [ ]:
eda.class_profile(train, io.NUMERIC_COLS, io.TARGET)

## Target

**Decision:** which metric behaviour to expect, and whether stratification is needed.

In [9]:
plots.target_overview(train, io.TARGET)

NotImplementedError: 

## Missingness

**Decision:** whether missingness is itself a feature, and what to impute.
The co-occurrence panel is the part that matters — do columns go missing together?

In [ ]:
plots.missingness(train)

## Numeric features

**Decision:** which features need transformation, and which are already informative.

In [ ]:
plots.numeric_grid(train, io.NUMERIC_COLS, target=io.TARGET)

## Categorical features

**Decision:** encoding strategy per column.

In [ ]:
plots.categorical_grid(train, io.CATEGORICAL_COLS, io.TARGET)

## Correlation

**Decision:** which redundant features to drop.

In [ ]:
plots.correlation(train, io.NUMERIC_COLS)

## Train vs test shift

**Decision:** whether the CV can be trusted at all. Companion to adversarial
validation — that gives one AUC, this shows which columns caused it.

In [ ]:
plots.train_test_shift(train, test, io.FEATURE_COLS)